In [0]:
from pyspark.sql.functions import max, min
#df.agg(max("tpep_pickup_datetime"), min("tpep_pickup_datetime")).display()

#df.agg(max("tpep_dropoff_datetime"), min("tpep_dropoff_datetime")).display()


In [0]:
from datetime import date
from dateutil.relativedelta import relativedelta

four_months_ago_start = date.today().replace(day=1) - relativedelta(months=4)
one_month_ago_start = date.today().replace(day=1) - relativedelta(months=1)



In [0]:
from pyspark.sql.functions import when, col, max, min, timestamp_diff
from pyspark.sql.types import StringType

df = spark.read.table("nyctaxi.01_bronze.yellow_trips_raw")

df = df.filter((col('tpep_pickup_datetime') >= four_months_ago_start) & (col('tpep_pickup_datetime') < one_month_ago_start))
#df = df.filter((col('tpep_dropoff_datetime') >= '2025-05-01') & (col('tpep_dropoff_datetime') < '2025-12-01'))

df = df.withColumn("vendor", 
                   when(df.VendorID == 1, "Creative Mobile Technologies, LLC").\
                    when(df.VendorID == 2, "Curb Mobility, LLC").\
                    when(df.VendorID == 6, "Myle Technologies Inc").\
                    when(df.VendorID == 7, "Helix").\
                    otherwise(None))

df = df.withColumn("rate_type",
                   when(col("RatecodeID") == 1, "Standard Rate").\
                    when(col("RatecodeID") == 2, "JFK").\
                    when(col("RatecodeID") == 3, "Newark").\
                    when(col("RatecodeID") == 4, "Nassau or Westchester").\
                    when(col("RatecodeID") == 5, "Negotiated Fare").\
                    when(col("RatecodeID") == 6, "Group Ride").\
                    otherwise("Unknown")
)

df = df.withColumn("payment_type",
                   when(col("payment_type") == 0, "Flex Fare trip").\
                    when(col("payment_type") == 1, "Credit card").\
                    when(col("payment_type") == 2, "Cash").\
                    when(col("payment_type") == 3, "No charge").\
                    when(col("payment_type") == 4, "Dispute").\
                    when(col("payment_type") == 6, "Voided trip").\
                    otherwise("Unknown"))

df = df.withColumn("trip_duration_minutes", (timestamp_diff("MINUTE", col("tpep_pickup_datetime"), col("tpep_dropoff_datetime")).cast("integer")))


df = df.select(
    col("vendor").alias("vendor"),
    col("tpep_pickup_datetime"),
    col("tpep_dropoff_datetime"),
    col("trip_duration_minutes"),
    col("passenger_count"),
    col("trip_distance"),
    col("rate_type"),
    col("store_and_fwd_flag"),
    col("PULocationID").alias("pu_location_id"),
    col("DOLocationID").alias("do_location_id"),
    col("payment_type"),
    col("fare_amount"),
    col("extra"),
    col("mta_tax"),
    col("tip_amount"),
    col("tolls_amount"),
    col("improvement_surcharge"),
    col("total_amount"),
    col("congestion_surcharge"),
    col("Airport_fee").alias("airport_fee"),
    col("cbd_congestion_fee"),
    col("ingesttime").alias("processed_timestamp"),
)

#df.filter( col("trip_duration_minutes") > 1440).sort(col("trip_duration_minutes").desc()).display()
# df.display()

df.write.mode("append").option("OverwriteSchema", "true").saveAsTable("nyctaxi.02_silver.yellow_trips_cleansed")

                    

In [0]:
spark.read.table("nyctaxi.02_silver.yellow_trips_cleansed").display()